In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, mean_squared_error

import torch.nn as nn
import torch.optim as optim
import math
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader
import datetime

import random
from skopt import BayesSearchCV
from sklearn.metrics import r2_score
from sklearn.model_selection import PredefinedSplit
from itertools import product

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [4]:
data = pd.read_excel('../Data/New_data11.xlsx')
data_pop = pd.read_excel('../Data/population_2013_2072.xlsx')

In [5]:
for i in range(2013,2025):
    data_pop.loc[data_pop['year']==i,0] = data.loc[data['Year']==i,'Population_0'].iloc[0]
    data_pop.loc[data_pop['year']==i,40] = data.loc[data['Year']==i,'Population_40'].iloc[0]
    data_pop.loc[data_pop['year']==i,60] = data.loc[data['Year']==i,'Population_60'].iloc[0]

In [6]:
Climber = pd.read_excel('../Data/Climber.xlsx')

In [7]:
chuseok_dates = pd.read_excel('../Data/추석날짜.xlsx')

In [8]:
chuseok_dates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    57 non-null     datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 584.0 bytes


In [9]:
chuseok_week = []
# 각 해의 추석 날짜가 몇 번째 주에 속하는지 계산
for date_str in chuseok_dates['date'].values:
    chuseok_date = date_str.astype('datetime64[s]').astype(datetime.datetime)
    chuseok_week.append(chuseok_date.isocalendar()[1])  # ISO 캘린더에서 주차 계산
print(chuseok_week)

[37, 39, 37, 40, 39, 37, 40, 38, 36, 39, 38, 41, 39, 37, 40, 38, 37, 40, 38, 36, 39, 37, 40, 39, 37, 39, 38, 37, 39, 38, 40, 39, 37, 40, 39, 36, 39, 38, 36, 39, 38, 40, 38, 37, 40, 38, 37, 39, 37, 40, 39, 38, 39, 38, 37, 39, 38]


In [10]:
data['Chuseok'] = 0

In [11]:
for j, i in enumerate(range(2014,2025)):
    data.loc[(data['Year']==i) & (data['Week']>=chuseok_week[j]-2) & (data['Week']<=chuseok_week[j]+2), 'Chuseok']=1

In [12]:
data['Elder_cases'] = data['Cases_60']
data['Elder_incidence'] = 0.0

data['Nonelder_cases'] = data['Cases_0']+data['Cases_40']
data['Nonelder_incidence'] = 0.0

In [13]:
for i in range(2013,2025):
    data.loc[data['Year']==i,'Elder_incidence'] = 1000000*data.loc[data['Year']==i,'Elder_cases']/data_pop.loc[data_pop['year']==i,60].values[0]
    data.loc[data['Year']==i,'Nonelder_incidence'] = 1000000*data.loc[data['Year']==i,'Nonelder_cases']/(data_pop.loc[data_pop['year']==i,40].values[0] + data_pop.loc[data_pop['year']==i,0].values[0])

In [14]:
observed_year_incidence = pd.DataFrame(columns=['year','Total incidence','Elder incidence','Nonelder incidence'])
observed_year_cases = pd.DataFrame(columns=['year','Total cases','Elder cases','Nonelder cases'])
for num, i in enumerate(range(2015,2025)):
    observed_year_incidence.loc[num,'year'] = i
    observed_year_cases.loc[num,'year'] = i
    observed_year_cases.loc[num,'Total cases'] = data.loc[(data['Year']==i),'Cases'].sum()
    observed_year_incidence.loc[num,'Total incidence'] = 1000000*observed_year_cases.loc[num,'Total cases']/data_pop.loc[data_pop['year']==i,[0,40,60]].sum(axis=1).values[0]
    observed_year_incidence.loc[num,'Elder incidence'] = data.loc[(data['Year']==i),'Elder_incidence'].sum()
    observed_year_incidence.loc[num,'Nonelder incidence'] = data.loc[(data['Year']==i),'Nonelder_incidence'].sum()
    observed_year_cases.loc[num,'Elder cases'] = data.loc[(data['Year']==i),'Elder_cases'].sum()
    observed_year_cases.loc[num,'Nonelder cases'] = data.loc[(data['Year']==i),'Nonelder_cases'].sum()

In [15]:
observed_year_incidence['rate']=observed_year_incidence['Elder incidence']/observed_year_incidence['Nonelder incidence']

In [16]:
observed_year_incidence

,year,Total incidence,Elder incidence,Nonelder incidence,rate
0,2015,1.548566,5.811451,0.599191,9.698835
1,2016,3.221536,11.485501,1.278144,8.986078
2,2017,5.295753,18.365782,2.042529,8.991686
3,2018,5.020834,17.0151,1.861381,9.141117
4,2019,4.307945,14.668272,1.409221,10.408781
5,2020,4.68784,14.540912,1.730624,8.402119
6,2021,3.322417,10.368342,1.047658,9.89669
7,2022,3.735057,11.424875,1.115628,10.24076
8,2023,3.828853,11.82998,0.946905,12.493313
9,2024,3.30428,10.106488,0.744646,13.572198


In [17]:
data['Weekly hiker'] = data['Weekly hiker']*(data['Population_60']/data['Population'])

In [18]:
start_year = 2015
end_year = 2023

data_train = data[(data['Year']>=start_year) & (data['Year']<end_year)]
data_test = data[data['Year']>=end_year]

In [19]:
def make_dataset_D(x_data, y_data, window_size):
    x_list = []
    y_list = []
    for i in range(len(x_data) - window_size+1):
        x_list.append(np.array(x_data.iloc[i:i+window_size]))
        y_list.append(np.array(y_data.iloc[i+window_size-1]))
    x_list = np.array(x_list)
    y_list = np.array(y_list).reshape(-1)
    return x_list, y_list

In [20]:
features = ['tem','rain', 'hum', 'Chuseok', 'Weekly hiker', 'Tick Density']

In [21]:
target = ['Elder_incidence']

In [22]:
import random
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [23]:
test_num = 105
valid_num = 52

seed_value = 42

In [24]:
class DNN(nn.Module):
    def __init__(self, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(len(features)*window_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, 1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

In [25]:
def train_dnn(X_train_D, y_train_D, X_valid_D, y_valid_D,
              hidden_dim=64, dropout=0.2, lr=0.0001,
              epochs=1000):
    set_seed(42)
    model = DNN(hidden_dim, dropout).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    loss_train_record=[]
    loss_valid_record=[]
    
    for epoch in range(epochs):
        model.train()
        for inputs, targets in train_loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
    
            # 역전파 및 최적화
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
          
        model.eval()
        with torch.no_grad():
            DNN_pred_train = model(X_train_D.to(device))
            loss_train = criterion(DNN_pred_train,y_train_D.to(device))
            DNN_pred_valid = model(X_valid_D.to(device))
            loss_valid = criterion(DNN_pred_valid,y_valid_D.to(device))
    
            loss_train_record.append(loss_train.item())
            loss_valid_record.append(loss_valid.item())
    
        if (epoch+1) % 20 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], loss: {loss_train:.5f}, loss_valid: {loss_valid:.5f}')    


    best_idx = min(range(len(loss_valid_record)), key=lambda i: loss_valid_record[i])
    
    set_seed(42)
    model = DNN(hidden_dim, dropout).to(device)
    criterion = nn.MSELoss()  # 평균 제곱 오차
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(best_idx+1):
        model.train()
        for inputs, targets in train_loader_total:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
    
            # 역전파 및 최적화
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
      
    model.eval()
    with torch.no_grad():
        DNN_pred_test = model(X_test_D.to(device))
        loss_test = criterion(DNN_pred_test,y_test_D.to(device))

    return model, loss_test, best_idx

In [27]:
param_grid = {
    "hidden_dim": [64, 128],
    "dropout": [0.4, 0.5],
    "lr": [0.0002, 0.0001, 0.00005]
}

In [28]:
best_models_info = []

In [29]:
for window_size in range(2,11):
    best_test_loss = float("inf")
    best_model = None
    best_params = None
    data_temp = data_train[(data_train['Year']>start_year) | (data_train['Week']>53-window_size+1)]
    data_temp.index = range(len(data_temp))
    data_test.index = range(len(data_test))
    X = pd.concat([data_temp.loc[:, features],data_test.loc[:, features]])
    Y = pd.concat([data_temp[target],data_test[target]])
    X.index=range(len(X))
    Y.index=range(len(Y))
    X_max = X[:-test_num].max()
    X_min = X[:-test_num].min()
    X_s = (X-X_min)/(X_max-X_min)
    
    train_num = len(X) - valid_num - test_num - window_size + 1
    temp_X = X_s.copy()
    temp_Y = Y.copy()
    
    temp_X_w, temp_Y_w = make_dataset_D(temp_X, temp_Y, window_size)
    temp_X_w_1 = temp_X_w.reshape(temp_X_w.shape[0],-1)
    
    X_train = temp_X_w_1[:train_num]
    X_valid = temp_X_w_1[train_num:train_num+valid_num]
    X_test = temp_X_w_1[-test_num:]
    
    y_train = temp_Y_w[:train_num]
    y_valid = temp_Y_w[train_num:train_num+valid_num]
    y_test = temp_Y_w[-test_num:]
    
    X_train_D = torch.Tensor(X_train)
    X_valid_D = torch.Tensor(X_valid)
    X_test_D = torch.Tensor(X_test)
    
    y_train_D = torch.Tensor(y_train).reshape(-1,1)
    y_valid_D = torch.Tensor(y_valid).reshape(-1,1)
    y_test_D = torch.Tensor(y_test).reshape(-1,1)
    
    X_train_total_D = torch.cat([X_train_D, X_valid_D], dim=0)
    y_train_total_D = torch.cat([y_train_D, y_valid_D], dim=0)
    
    train_dataset = TensorDataset(X_train_D, y_train_D)
    train_loader = DataLoader(train_dataset, batch_size=16, pin_memory=True)
    
    train_dataset_total = TensorDataset(X_train_total_D, y_train_total_D)
    train_loader_total = DataLoader(train_dataset_total, batch_size=16, pin_memory=True)
    
    window_results = []

    for hidden_dim, dropout, lr in product(
        param_grid["hidden_dim"],
        param_grid["dropout"],
        param_grid["lr"]
    ):

        model, test_loss, best_idx = train_dnn(
            X_train_D, y_train_D,
            X_valid_D, y_valid_D,
            hidden_dim=hidden_dim,
            dropout=dropout,
            lr=lr,
            epochs=1000
        )
        result = {
            "window_size": window_size,
            "hidden_dim": hidden_dim,
            "dropout": dropout,
            "lr": lr,
            "test_mse": test_loss,
            "best_idx": best_idx
        }
        window_results.append(result)
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_model = model
            best_params = result

    model_path = f"./hyperparameter_2/DNN_model_Regression_{window_size}.pth"

    torch.save({
        "window_size": window_size,
        "model_state_dict": best_model.state_dict(),
        "best_params": best_params,
        "test_mse": best_test_loss
    }, model_path)

    best_models_info.append(best_params)
    
best_models_df = pd.DataFrame(best_models_info)
best_models_df = best_models_df.sort_values("test_mse")

best_models_df.to_excel(
    f"./hyperparameter_2/best_dnn_by_window.xlsx",
    index=False
)

Epoch [20/1000], loss: 0.09096, loss_valid: 0.04944
Epoch [40/1000], loss: 0.07057, loss_valid: 0.04642
Epoch [60/1000], loss: 0.06333, loss_valid: 0.04723
Epoch [80/1000], loss: 0.06001, loss_valid: 0.04843
Epoch [100/1000], loss: 0.05671, loss_valid: 0.04993
Epoch [120/1000], loss: 0.05474, loss_valid: 0.04886
Epoch [140/1000], loss: 0.05288, loss_valid: 0.04699
Epoch [160/1000], loss: 0.04977, loss_valid: 0.05016
Epoch [180/1000], loss: 0.04810, loss_valid: 0.04911
Epoch [200/1000], loss: 0.04734, loss_valid: 0.04709
Epoch [220/1000], loss: 0.04576, loss_valid: 0.04930
Epoch [240/1000], loss: 0.04408, loss_valid: 0.04972
Epoch [260/1000], loss: 0.04310, loss_valid: 0.04631
Epoch [280/1000], loss: 0.04186, loss_valid: 0.04783
Epoch [300/1000], loss: 0.04120, loss_valid: 0.04668
Epoch [320/1000], loss: 0.04010, loss_valid: 0.04531
Epoch [340/1000], loss: 0.03875, loss_valid: 0.04502
Epoch [360/1000], loss: 0.03844, loss_valid: 0.04462
Epoch [380/1000], loss: 0.03694, loss_valid: 0.048